# Integration Testing Assignment

**Task:** Set up an environment for integration testing and write test cases that validate the
interaction between different components of a system.

**System under test:** A small Flask API backed by a SQLite database (two components: the
*web layer* and the *data layer*), tested end-to-end with `pytest`.

## 1. Environment Setup

In [1]:
setup_instructions = r'''
python -m venv venv
source venv/bin/activate          # Windows: venv\Scripts\activate
pip install flask flask_sqlalchemy pytest requests
'''
print(setup_instructions)



python -m venv venv
source venv/bin/activate          # Windows: venv\Scriptsctivate
pip install flask flask_sqlalchemy pytest requests



<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_695/3989609795.py:1: SyntaxWarning: invalid escape sequence '\S'
  setup_instructions = '''


## 2. Application Under Test (API + DB)

In [2]:
import os
os.makedirs('project', exist_ok=True)

with open('project/app.py', 'w') as f:
    f.write('''from flask import Flask, request, jsonify
from flask_sqlalchemy import SQLAlchemy

def create_app(db_uri="sqlite:///users.db"):
    app = Flask(__name__)
    app.config["SQLALCHEMY_DATABASE_URI"] = db_uri
    db = SQLAlchemy(app)

    class User(db.Model):
        id = db.Column(db.Integer, primary_key=True)
        name = db.Column(db.String(80), nullable=False)
        email = db.Column(db.String(120), unique=True, nullable=False)

    with app.app_context():
        db.create_all()

    @app.route("/users", methods=["POST"])
    def create_user():
        data = request.get_json()
        user = User(name=data["name"], email=data["email"])
        db.session.add(user)
        db.session.commit()
        return jsonify({"id": user.id, "name": user.name, "email": user.email}), 201

    @app.route("/users/<int:user_id>", methods=["GET"])
    def get_user(user_id):
        user = User.query.get(user_id)
        if not user:
            return jsonify({"error": "not found"}), 404
        return jsonify({"id": user.id, "name": user.name, "email": user.email})

    @app.route("/users/<int:user_id>", methods=["DELETE"])
    def delete_user(user_id):
        user = User.query.get(user_id)
        if not user:
            return jsonify({"error": "not found"}), 404
        db.session.delete(user)
        db.session.commit()
        return "", 204

    app.db = db
    return app

if __name__ == "__main__":
    create_app().run(debug=True)
''')
print("Created project/app.py (Flask + SQLAlchemy)")


Created project/app.py (Flask + SQLAlchemy)


## 3. Integration Test Cases
These tests exercise the **real Flask app talking to a real (temp) SQLite database** — i.e. they validate the interaction between the API layer and the persistence layer, which is what makes them *integration* tests rather than unit tests.

In [3]:
import os
os.makedirs('project/tests', exist_ok=True)

with open('project/tests/test_integration.py', 'w') as f:
    f.write('''import os
import pytest
from app import create_app

@pytest.fixture
def client(tmp_path):
    db_file = tmp_path / "test.db"
    app = create_app(db_uri=f"sqlite:///{db_file}")
    app.config["TESTING"] = True
    with app.test_client() as client:
        yield client

def test_create_and_get_user(client):
    resp = client.post("/users", json={"name": "Alice", "email": "alice@example.com"})
    assert resp.status_code == 201
    user_id = resp.get_json()["id"]

    resp2 = client.get(f"/users/{user_id}")
    assert resp2.status_code == 200
    assert resp2.get_json()["name"] == "Alice"

def test_get_nonexistent_user(client):
    resp = client.get("/users/999")
    assert resp.status_code == 404

def test_delete_user(client):
    resp = client.post("/users", json={"name": "Bob", "email": "bob@example.com"})
    user_id = resp.get_json()["id"]

    del_resp = client.delete(f"/users/{user_id}")
    assert del_resp.status_code == 204

    get_resp = client.get(f"/users/{user_id}")
    assert get_resp.status_code == 404

def test_duplicate_email_fails_gracefully(client):
    client.post("/users", json={"name": "Carl", "email": "dup@example.com"})
    resp = client.post("/users", json={"name": "Carl2", "email": "dup@example.com"})
    # Uniqueness constraint on email should surface as a server/client error, not a silent success
    assert resp.status_code != 201
''')
print("Created project/tests/test_integration.py")
print("Run with: cd project && pytest -v")


Created project/tests/test_integration.py
Run with: cd project && pytest -v


## 4. Wiring Into CI (optional but recommended)

In [4]:
ci_yaml = '''name: Integration Tests

on: [push, pull_request]

jobs:
  integration-tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install flask flask_sqlalchemy pytest requests
      - run: cd project && pytest -v
'''
import os
os.makedirs('project/.github/workflows', exist_ok=True)
with open('project/.github/workflows/integration.yml', 'w') as f:
    f.write(ci_yaml)
print("Created project/.github/workflows/integration.yml")


Created project/.github/workflows/integration.yml
